# M5A1 - Inspeção Visual de Itens em Esteira de Manufatura

Na prática de hoje vamos refinar um modelo para a tarefa de inspeção visual.

Esse notebook está estruturado da seguinte forma.

- Introdução
- Carregar Base de Dados
- Refinar Modelo
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [ ]:
!pip install torch torchvision huggingface_hub ultralytics

Importar as bibliotecas

In [23]:
import yaml
import os

from ultralytics import YOLO
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil
from IPython.display import Video

## Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

Referência: https://huggingface.co/datasets/johnatanvq/fruits-dataset

In [24]:
# 1) Baixar somente o subdiretório fruitsData/
local_repo_dir = snapshot_download(
    repo_id="johnatanvq/fruits-dataset",
    repo_type="dataset",
    max_workers=1,
    resume_download=True,  
    allow_patterns=["fruitsData/**"],  # baixa só essa pasta
)

print("Arquivos baixados em:", local_repo_dir)

# 2) Mover/copiar para uma pasta final estilo ImageFolder (se quiser customizar o caminho)
src = Path(local_repo_dir) / "fruitsData"
dst = Path("data/fruits")  # pasta final onde você quer o ImageFolder

# Copiando arquivos para dst.
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print("ImageFolder pronto em:", dst)


Fetching 322 files:  19%|█▉        | 62/322 [00:18<01:40,  2.60it/s]'[WinError 10054] Foi forçado o cancelamento de uma conexão existente pelo host remoto' thrown while requesting HEAD https://huggingface.co/datasets/johnatanvq/fruits-dataset/resolve/7cf7e5de00d186e99fe46357a80be02f3e110a34/fruitsData/images/5a659c4f-fruit_77.jpg
Retrying in 1s [Retry 1/5].
Fetching 322 files:  21%|██▏       | 69/322 [00:48<02:57,  1.43it/s]


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
  # Lê o arquivos "classes.txt".
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Cria o dicionário a ser salvo.
  data = {
      'path': 'data/fruits',
      'train': 'images',
      'val': 'images',
      'nc': number_of_classes,
      'names': classes
  }

  # Escreve o arquivo YAML.
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Chama a função.
create_data_yaml("data/fruits/classes.txt", "yolo_train.yaml")

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()

In [ ]:
# Carrega o modelo pré-treinado.
model = YOLO("yolo11n.pt")

# Treina o modelo utilizando as informações do arquivo YAML.
# Definimos também a quantidade de épocas, o batch, e o tamanho das imagens.

#results = model.train(data="./yolo_train.yaml", project="praticas/modulo_5/aula_1", epochs=10, batch=2, imgsz=480)

results = model.train(data="./yolo_train.yaml", project=str(BASE_DIR / "runs"), epochs=10, batch=2, imgsz=480)

In [ ]:
# Video pode ser necessário alterar o path.
Video("video.mov")


In [ ]:
import os

print(os.getcwd())

In [ ]:
#model.predict("video.mov", save=True, project="praticas/modulo_5/aula_1")
#model.predict("video.mov", save=True, project="predict")

results = model.predict("video.mov", project=os.getcwd(), save=True)

In [ ]:
print(results[0].save_dir)

In [ ]:
#Video("/home/joaoferreira/trilha_visao_computacional/praticas/modulo_5/aula_1/praticas/modulo_5/aula_1/predict/video.avi")
#Video("predict/video.avi")

Video(f"{results[0].save_dir}/video.avi")

## Próximos Passos e Referências

Nas próximas práticas vamos continuar trabalhando com problemas reais que envolvem Visão Computacional.

Uma lista não exaustiva de referências segue:

- https://docs.ultralytics.com/modes/train/
- https://docs.ultralytics.com/modes/predict/
- https://huggingface.co/datasets/johnatanvq/fruits-dataset
- https://huggingface.co/datasets
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Tente alterar a base de dados e veja se o modelo continua funcionando?
- [ ] Tente alterar alguns hiperparâmetros de treinamento, batch e resolução da imagens e veja como isso altera os resultados.